<a href="https://colab.research.google.com/github/soule-geophysics/geol-333-714/blob/main/notebooks/HW5a_slopes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> **Do this first: File > Save a copy in Drive.**
> You are viewing a shared notebook. Anything you type here is NOT saved.
> Save your own copy now, work only in the copy, and rename it
> `HW5a_LASTNAME.ipynb` (exact filename and due date in the assignment
> calendar on Brightspace).

# HW5a: Layer Velocities from Travel-Time Slopes

**Course:** GEOL 333 / 714, Geophysical Exploration Methods, Fall 2026
**Due:** Sunday, December 13, 2026, 11:59 PM
**Submit:** This notebook (`.ipynb`) to the Brightspace HW5a dropbox

**Extra credit:** This notebook is optional. It is graded out of 100 and counts as a bonus of up to 50 points in Problem Sets & Labs.

This notebook is new, and you are the first class to work it.

The mechanics of Colab (saving, running cells, errors, uploading) are on the **Working in Colab** page in the Resources module on Brightspace.

This assignment follows the Wednesday, November 11 class (Burger Chapter 2 and section 3.1).

## What you will do

1. Load a refraction survey: the first-arrival time at each of 24 geophones.
2. Plot travel time against distance and find the two straight lines in it.
3. Choose where the steep line ends, then fit each line. The slope of each line is one over a velocity.
4. Find the distance where the two lines cross.

## What you will hand in

This same notebook with every cell run and the answer cells filled in. Each question has one answer cell that opens with its label. Double-click the cell and type under the label.

## Feedback

After each section there is a feedback cell, and one more at the end. Tell me what worked, what did not, and where you got stuck. Write it as you go, while it is fresh. The feedback cells carry no points.

## Learning objectives

This notebook builds these course objectives:

2. Use Snell's law to find the angle of a ray refracted at the boundary between two layers. Compute the critical angle from the two velocities. State Huygens' principle: every point on a wave front is a source of new waves, and the next wave front is the line that touches them all. Use it to explain where the head wave comes from. Explain why a head wave forms only when the lower layer is faster.
3. Sketch the direct wave and the head wave on a time-distance plot. Mark the crossover distance, where the head wave becomes the first arrival.
4. Find each layer's velocity as one over the slope of its line on a time-distance plot. The direct wave is a line through the origin with slope 1/V1. The head wave is a line with slope 1/V2.
5. Make a time-distance plot from first-break times. Fit a straight line to each segment.

## Data source and citation

> Refraction dataset (synthetic): generated by a course script (GEOL 333/714 course materials, seed 333). Two layers over a flat interface, with a direct wave, a head wave and a small random error added to each time, like the error in picking a time from a real record. No external license.

A [**geophone**](https://github.com/soule-geophysics/geol-333-714/blob/main/GLOSSARY.md#geophone) is a sensor set in the ground that records ground motion. The survey is one line of 24 geophones, 3 m apart, from 3 m to 72 m. The source is a hammer blow at 0 m. At each geophone, the script computes the time of the first ground motion and adds a small random error. That time is the [**first break**](https://github.com/soule-geophysics/geol-333-714/blob/main/GLOSSARY.md#first-break). The data are generated by a script, so no measurement was made at a real site.

| Column | Meaning |
|---|---|
| `offset_m` | Distance from the source to the geophone (m) |
| `first_break_ms` | First-break time (ms) |
| `travel_time_s` | The same time in seconds |

**A first break on a seismic record.** The figure below is a synthetic record computed from the same two-layer model that generated this notebook's data, with the same 24 geophones. Each wavy line is the ground motion at one geophone, drawn at that geophone's distance from the source. Time increases up the page, as on the time-distance plot later in this notebook. Every geophone past the first records two waves, arriving at different times. The short orange bars mark the first break on each trace, where the ground first moves. They are the first-break times in `refraction.csv`.

> Synthetic record: generated by a course script (GEOL 333/714 course materials, seed 333) from the model behind `refraction.csv`, with a 150 Hz wavelet and small random noise added to each trace. No external license.

![Synthetic wiggle traces from the HW5a model with the first breaks marked](https://raw.githubusercontent.com/soule-geophysics/geol-333-714/main/figures/geol333_hw5a_synthetic_gather_first_breaks.png)

**Figure description:** One wavy trace per geophone, distance from the source (3 m to 72 m, x-axis) against time after the hammer blow (0 to 150 ms, y-axis, increasing upward). Filled black lobes are ground motion in one direction. The trace at 3 m carries one arrival; every other trace carries two. One set of arrivals falls on a steep straight line, from about 6 ms at 3 m to about 144 ms at 72 m. The other falls on a shallower straight line, from about 34 ms at 6 m to about 71 ms at 72 m. The two lines cross a little past 20 m. Short orange bars mark the first break on all 24 traces. They follow the steep line near the source and the shallow line beyond the crossing.

## Loading the data

The code below loads `refraction.csv` automatically from a stable public web address, so in most cases you do not need to download anything: run the cells.

**If you have no internet, or the link is not live yet,** use the Brightspace fallback:

1. Download `refraction.csv` from the Brightspace HW5a page.
2. In Google Colab, click the **Files** icon in the left sidebar.
3. Drag `refraction.csv` into the file panel.
4. In the loading cell below, comment out the `pd.read_csv(DATA_URL)` line and use the commented `pd.read_csv("refraction.csv")` line instead.

> **Colab deletes uploaded files when the runtime disconnects.**
> If a CSV you uploaded by hand has vanished, re-run the data-loading cell
> (the URL load restores the data) or re-upload the file. Code and written
> answers persist in your own saved copy.

## Setup

These imports load NumPy (numbers and arrays), Pandas (tables), and Plotly (interactive plots). All three come pre-installed in Colab; no `pip install` needed.

The cell also defines `run_check`. A run-check cell prints one line that reads PASS or FAIL. A run check carries no question and no points. It confirms that the cells above ran as written. If one reads FAIL, the problem is upstream of it.

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# Course accessibility default: colorblind-safe qualitative palette
px.defaults.color_discrete_sequence = px.colors.qualitative.Safe
COLORS = px.colors.qualitative.Safe


def run_check(name, passed, detail):
    """Print one run-check line: PASS, or FAIL with the value that missed."""
    print(f'{name} {"PASS" if passed else "FAIL"}: {detail}')


## Velocities from the slopes of the direct and head-wave lines

At each geophone, the first break comes from one of two waves (Burger sections 2.2.6 and 2.2.8, Figs 2.20 and 2.21).

The [**direct wave**](https://github.com/soule-geophysics/geol-333-714/blob/main/GLOSSARY.md#direct-wave) travels along the surface at the velocity of the upper layer, $V_1$. Its travel time grows in step with distance (Burger eq 3.2):

$$t = \frac{x}{V_1}$$

The [**head wave**](https://github.com/soule-geophysics/geol-333-714/blob/main/GLOSSARY.md#head-wave) travels down to the interface and meets it at the **critical angle** (Burger eq 2.27):

$$\theta_{ic} = \sin^{-1}\left(\frac{V_1}{V_2}\right)$$

It then runs along the interface at the velocity of the lower layer, $V_2$, and sends energy back up to every geophone it passes (Fig 2.16).

On a plot of time against distance, each wave makes a straight line. The slope of the line, written $dt/dx$, is the change in time for each meter of distance. It is one over the velocity (Burger eq 3.3 to 3.5):

$$\frac{dt}{dx} = \frac{1}{V_1} \qquad V_1 = \frac{1}{\text{slope}}$$

The head-wave line gives $V_2$ the same way.

**Load.** Reads `refraction.csv` into `df` and defines `x`, the distance from the source, and `t`, the first-break time in seconds. The printout gives the number of geophones, the range of distances and the spacing.

In [ ]:
# Primary path: load directly from a stable public URL (no upload needed).
DATA_URL = "https://raw.githubusercontent.com/soule-geophysics/geol-333-714/main/data/refraction.csv"
df = pd.read_csv(DATA_URL)

# Fallback (no internet, or URL not live yet): download refraction.csv from the
# Brightspace HW5a page, drag it into the Colab file panel, then comment out the
# df = pd.read_csv(DATA_URL) line above and use:
# df = pd.read_csv("refraction.csv")

x = df['offset_m'].to_numpy()          # distance from the source (m)
t = df['travel_time_s'].to_numpy()     # first-break time (s)
print(f'{len(df)} geophones, {x.min():.0f} m to {x.max():.0f} m, '
      f'{np.diff(x).min():.0f} m apart')


**Plot the first breaks.** This cell plots first-break time against distance, then prints the same numbers as a table.

In [ ]:
fig = px.scatter(df, x='offset_m', y='first_break_ms',
                 labels={'offset_m': 'Distance from the source (m)',
                         'first_break_ms': 'First-break time (ms)'},
                 title='First breaks against distance')
fig.update_traces(marker=dict(size=9, symbol='circle'))
fig.show()

# Non-visual path: every geophone as a table.
print(df[['offset_m', 'first_break_ms']].to_string(index=False))


**Figure description:** A scatter plot of first-break time (ms, y-axis) against distance from the source (m, x-axis), one circle per geophone. The times run from about 6 ms at 3 m to about 71 ms at 72 m. The points fall on two straight lines: a steep one near the source and a shallower one farther out. Non-visual path: the table printed under the plot lists every geophone. Where the time added from one geophone to the next drops, the points have moved from the steep line to the shallow one.

**Question 1.** At the geophones near the source, is the first break the direct wave or the head wave? At the far geophones, which one is it? The head wave travels a longer path. Why can it still arrive first at the far geophones? (Burger section 2.2.8, Figs 2.20 and 2.21.)

**Q1 answer:**



**Feedback on this section.** What worked? What did not? Where did you get stuck?

**Feedback 1:**



**Fit the two lines.** You set the split. `LAST_DIRECT_M` is the distance of the last geophone on the steep line. Read it from the plot and the table above: count the last point that still sits on the steep straight line that starts at the source. The first point past it falls below that line. On the first line of the next cell, replace `None` with that distance in meters, then run the cell.

The cell fits one straight line to the geophones up to `LAST_DIRECT_M`, the direct wave, and another to the geophones beyond it, the head wave. `np.polyfit` with `cov=True` gives each slope with its 1-sigma uncertainty, as in HW0 to HW2. The 1-sigma uncertainty is one standard deviation: the true slope lies within one sigma of the fitted slope about two times in three. Each velocity is one over its slope. The cell also prints the critical angle from eq 2.27 and the distance where the two fitted lines cross.

In [ ]:
LAST_DIRECT_M = None   # replace None with the distance (m) of the last geophone on the steep line

if LAST_DIRECT_M is None:
    raise ValueError('Set LAST_DIRECT_M on the first line of this cell, then run the cell again.')

direct = x <= LAST_DIRECT_M
head = x > LAST_DIRECT_M

# One straight line through each group. With cov=True, np.polyfit also returns a
# small table that says how well the data fix the slope and the intercept. The
# square root of its [0, 0] entry is the 1-sigma uncertainty on the slope.
(slope_d, intercept_d), cov_d = np.polyfit(x[direct], t[direct], 1, cov=True)
(slope_h, intercept_h), cov_h = np.polyfit(x[head], t[head], 1, cov=True)

# Velocity is one over the slope. The slope uncertainty carries over as sigma_s / s**2.
v1 = 1.0 / slope_d
v2 = 1.0 / slope_h
v1_sigma = np.sqrt(cov_d[0, 0]) / slope_d**2
v2_sigma = np.sqrt(cov_h[0, 0]) / slope_h**2

theta_ic = np.degrees(np.arcsin(v1 / v2))                  # critical angle, Burger eq 2.27
x_co = (intercept_h - intercept_d) / (slope_d - slope_h)   # where the two lines give the same time

print(f'geophones on the direct line: {direct.sum()}; on the head-wave line: {head.sum()}')
print(f'direct-wave slope: {slope_d * 1000:.3f} ms per m   V1 = {v1:.0f} +/- {v1_sigma:.0f} m/s (1 sigma)')
print(f'head-wave slope:   {slope_h * 1000:.3f} ms per m   V2 = {v2:.0f} +/- {v2_sigma:.0f} m/s (1 sigma)')
print(f'critical angle:    {theta_ic:.1f} degrees')
print(f'the lines cross at {x_co:.1f} m')


This cell plots the first breaks with the two fitted lines and marks where the lines cross.

In [ ]:
x_line = np.linspace(0.0, x.max(), 200)
fig = go.Figure()
fig.add_scatter(x=x, y=t * 1000, mode='markers', name='first breaks',
                marker=dict(size=9, symbol='circle', color=COLORS[0]))
fig.add_scatter(x=x_line, y=(slope_d * x_line + intercept_d) * 1000, mode='lines',
                name=f'direct-wave line, V1 = {v1:.0f} m/s',
                line=dict(color=COLORS[3], width=2, dash='solid'))
fig.add_scatter(x=x_line, y=(slope_h * x_line + intercept_h) * 1000, mode='lines',
                name=f'head-wave line, V2 = {v2:.0f} m/s',
                line=dict(color=COLORS[4], width=2, dash='dash'))
fig.add_vline(x=x_co, line_dash='dot', annotation_text=f'lines cross at {x_co:.1f} m')
fig.update_xaxes(title_text='Distance from the source (m)')
fig.update_yaxes(title_text='First-break time (ms)', rangemode='tozero')
fig.update_layout(title='First breaks with the two fitted lines')
fig.show()


**Figure description:** The first breaks as circles, first-break time (ms, y-axis) against distance from the source (m, x-axis), with two fitted lines drawn from 0 m to 72 m. A solid line runs through the near points and starts at zero time at the source. A dashed line runs through the far points and meets the time axis at about 30 ms. The solid line is the steeper of the two. A dotted vertical line marks where the two lines cross, near 21 m. Non-visual path: the previous cell prints both slopes, both velocities and the crossing distance.

**Run check 1.** This check reads PASS when the two velocities and the crossing distance fall in the range a good split gives. If it reads FAIL, look again at the plot and the table, change `LAST_DIRECT_M` in the fit cell, and run the fit cell and every cell after it again. If it still reads FAIL, choose Runtime > Restart and run all.

In [ ]:
run_check('Run check 1',
          480 <= v1 <= 525 and 1650 <= v2 <= 1850 and 19 <= x_co <= 23,
          f'V1 {v1:.0f} m/s; V2 {v2:.0f} m/s; lines cross at {x_co:.1f} m')


**Question 2.** Report $V_1$ and $V_2$ with their uncertainties. Report the critical angle. Which layer is faster? Try eq 2.27 with the lower layer slower than the upper one. What happens to the sine of the critical angle? What does that mean for the head wave? (Burger section 2.2.6.)

**Q2 answer:**



**Question 3.** The two lines cross at the [**crossover distance**](https://github.com/soule-geophysics/geol-333-714/blob/main/GLOSSARY.md#crossover-distance). Past it, the head wave gives the first break. Suppose the interface were deeper, with the same two velocities. Would the crossover move toward the source or away from it? Explain from the ray paths in Fig 2.20.

**Q3 answer:**



**Feedback on this section.** What worked? What did not? Where did you get stuck?

**Feedback 2:**



**Feedback on the whole notebook.** What would you change before another class works this notebook?

**Feedback 3:**



## How to submit

1. Run all cells from the top. (`Runtime > Run all` in Colab.) Every run-check line reads PASS.
2. Make sure every answer cell is filled in.
3. `File > Download > Download .ipynb`.
4. Rename the file to `HW5a_LASTNAME.ipynb` (for example `HW5a_Smith.ipynb`).
5. Upload it to the Brightspace HW5a dropbox by **Sunday, December 13, 11:59 PM**.

If something is broken or unclear, post on the **Ask the Class (General Q&A)** discussion topic. Other students may have the same question.